# 19 — Paper Figures: Dataset Construction and Multiclass Distribution

Two figures for the paper's Dataset section, built for the LaTeX merge /
figure-integration pass:

1. **Dataset construction** (`dataset_overview.pdf/png`) — a 10-channel
   schematic of one real event (OMNI + GOES, 24h pre-onset), in the same
   style as `../SEP_DataAugmentation 2/paper/figures/dataset_overview.png`
   (that project's Figure 1). Reuses its own generation code from
   `8_Figures.ipynb`, adapted to read this project's `./dataset/*.csv`.
2. **Multiclass distribution** (`multiclass_distribution.pdf/png`) — sample
   counts per original label (NSEP, gt10, gt30, gt60, gt100) before the
   gt10/gt30/gt60/gt100 labels are collapsed into a single SEP class.

**Reads:** `./dataset/*.csv` (read-only, the raw per-channel files).
**Writes:** `./paper/figures/dataset_overview.pdf/.png` and
`./paper/figures/multiclass_distribution.pdf/.png`.
**Does not touch:** notebooks 11-18 or their output.

Note: notebook 14 previously wrote a "Class Imbalance Across Splits" bar
chart to `dataset_overview.pdf/png` (train/val/test counts, not the raw
channel schematic). That figure has been renamed to
`dataset_splits.pdf/png` (see notebook 14) so this filename is free for
the actual dataset-construction schematic, matching the naming used in
SEP_DataAugmentation 2.


## 1. Style (same as notebooks 14-18)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = "./paper/Figures"
DATASET_DIR = "./dataset"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.4, "ytick.major.size": 2.4,
    "xtick.labelsize": 6.4, "ytick.labelsize": 6.4,
    "axes.labelsize": 7.0, "axes.titlesize": 7.4, "legend.fontsize": 6.2,
    "axes.edgecolor": "0.35", "text.color": "0.0",
    "axes.labelcolor": "0.0", "xtick.color": "0.25", "ytick.color": "0.25",
})

BLUE, VERM, TEAL, GOLD = "#0072B2", "#D55E00", "#009E73", "#CC9A00"
GREY, LG = "0.45", "#DCDCDC"

print("Style ready. Dataset dir:", os.path.abspath(DATASET_DIR))


Style ready. Dataset dir: /Users/samskanderi/Documents/Claude/SEP_DataAugmentation/dataset


## 2. Dataset Construction Figure

Reads one real gt100 event (row 9391 of the source catalog, the same event `SEP_DataAugmentation 2`'s own Figure 1 uses) directly from the ten raw per-channel CSVs. Only that one row is loaded from each file, so this stays fast despite each CSV being hundreds of MB.

In [2]:
"""Fig. — dataset construction: 10-channel schematic of one real SEP event.

Adapted from SEP_DataAugmentation 2's Figure 1 generation code
(8_Figures.ipynb, cell "Figure 1 -- Dataset Construction"), re-pointed at
this project's own ./dataset/*.csv (same underlying OMNI+GOES catalog:
17,794 events, event #9391 is a >100 MeV SEP event in both projects).
"""
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib

EVENT = 9391  # a >100 MeV SEP event from the catalog

OMNI = ["V", "Vx", "Np", "Tp", "F"]
GOES = ["Xl", "Xs", "P4", "P5", "P6"]
LOG = {"P4", "P5", "P6", "Xl", "Xs"}
PRETTY = {"V": "$V$", "Vx": "$V_x$", "Np": "$N_p$", "Tp": "$T_p$", "F": "$F$",
          "Xl": "Xl", "Xs": "Xs", "P4": "P4", "P5": "P5", "P6": "P6"}

FEATS = ["F", "Np", "P4", "P5", "P6", "Tp", "V", "Vx", "Xl", "Xs"]
series = {}
for f in FEATS:
    df = pd.read_csv(f"{DATASET_DIR}/{f}.csv",
                      skiprows=lambda i: i > 0 and i != EVENT + 1,
                      low_memory=False)
    series[f] = df.iloc[0, -288:].apply(pd.to_numeric,
                                         errors="coerce").values.astype(float)
X = np.stack([series[f] for f in FEATS], axis=-1)
fidx = {f: i for i, f in enumerate(FEATS)}
t = np.arange(288) * 5 / 60 - 24

fig = plt.figure(figsize=(3.45, 4.05))
gs = GridSpec(10, 1, figure=fig, hspace=0.0,
              left=0.175, right=0.97, top=0.935, bottom=0.215)

order = OMNI + GOES
axes = []
for k, f in enumerate(order):
    ax = fig.add_subplot(gs[k, 0])
    v = X[:, fidx[f]].astype(float)
    color = BLUE if f in OMNI else VERM
    if f in LOG:
        ax.set_yscale("log")
        v = np.where(v > 0, v, np.nan)
    ax.plot(t, v, lw=0.6, color=color, solid_joinstyle="round")
    ax.set_xlim(-24, 0)
    ax.set_yticks([])
    ax.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax.tick_params(axis="y", which="both", left=False, right=False, length=0,
                    labelleft=False)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.4)
    ax.spines["bottom"].set_color("0.78")
    ax.text(-0.028, 0.5, PRETTY[f], transform=ax.transAxes, ha="right",
             va="center", fontsize=7.2, color=color)
    ax.axvline(0, color="0.35", lw=0.7, ls=(0, (2.2, 1.6)), zorder=0)
    if k < 9:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
    axes.append(ax)

ax_last = axes[-1]
ax_last.set_xticks([-24, -18, -12, -6, 0])
ax_last.set_xticklabels(["$-24$", "$-18$", "$-12$", "$-6$", "0"], fontsize=6.6)
ax_last.spines["bottom"].set_color("0.3")
ax_last.spines["bottom"].set_linewidth(0.5)
ax_last.set_xlabel("hours before flare onset   (288 steps, 5-min cadence)",
                    fontsize=6.9, labelpad=2.0)

axes[0].text(-0.4, 1.10, "flare onset",
             transform=axes[0].get_xaxis_transform(),
             ha="right", va="bottom", fontsize=6.6, color="0.35")
axes[0].annotate("", xy=(0, 1.13), xytext=(-0.35, 1.13),
                  xycoords=axes[0].get_xaxis_transform(),
                  arrowprops=dict(arrowstyle="-|>", lw=0.6, color="0.35",
                                  shrinkA=0, shrinkB=0))


def bracket(ax_top, ax_bot, label, color):
    x = 0.048
    y0 = ax_bot.get_position().y0
    y1 = ax_top.get_position().y1
    fig.add_artist(Line2D([x, x], [y0, y1], color=color, lw=0.9))
    for y in (y0, y1):
        fig.add_artist(Line2D([x, x + 0.015], [y, y], color=color, lw=0.9))
    fig.text(x - 0.014, (y0 + y1) / 2, label, rotation=90, ha="center",
              va="center", fontsize=7.4, color=color)


fig.canvas.draw()
bracket(axes[0], axes[4], "OMNI", BLUE)
bracket(axes[5], axes[9], "GOES", VERM)

axb = fig.add_axes([0.055, 0.018, 0.915, 0.105])
axb.set_xlim(0, 1)
axb.set_ylim(0, 1)
axb.set_xticks([])
axb.set_yticks([])
for s in axb.spines.values():
    s.set_linewidth(0.5)
    s.set_color("0.80")
axb.set_facecolor("none")

axb.text(0.5, 0.68,
         r"stack 10 channels  $\Rightarrow$  "
         r"$\mathbf{X}\in\mathbb{R}^{17{,}794\times288\times10}$",
         ha="center", va="center", fontsize=7.4)
axb.text(0.5, 0.24,
         "labels:   17,625 NSEP   and   169 SEP        (104 : 1)",
         ha="center", va="center", fontsize=6.9, color="0.0")

fig.savefig(f"{FIG_DIR}/dataset_overview.pdf")
fig.savefig(f"{FIG_DIR}/dataset_overview.png", dpi=400)
plt.show()
print("Saved dataset_overview.pdf/png (event #9391, gt100)")


Saved dataset_overview.pdf/png (event #9391, gt100)


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_47094/4219060999.py:115: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Multiclass Distribution Figure

Sample counts per *original* label, before the four SEP subtypes (gt10/gt30/gt60/gt100, i.e. $\geq$10/30/60/100 MeV proton-flux thresholds) are collapsed into one binary SEP class. Counted directly from the `Label` column of `dataset/V.csv` (every per-channel CSV shares the same label column and row order).

In [3]:
# FIG -- sample counts per original multiclass label, before binary collapse
labels_df = pd.read_csv(f"{DATASET_DIR}/V.csv", usecols=["Label"])
counts = labels_df["Label"].value_counts()

order = ["NSEP", "gt10", "gt30", "gt60", "gt100"]
pretty = {"NSEP": "Non-SEP", "gt10": r"$\geq$10 MeV", "gt30": r"$\geq$30 MeV",
          "gt60": r"$\geq$60 MeV", "gt100": r"$\geq$100 MeV"}
vals = [int(counts.get(k, 0)) for k in order]
colors = ["0.62"] + [TEAL] * 4

fig, ax = plt.subplots(figsize=(3.4, 2.6))
x = np.arange(len(order))
ax.bar(x, vals, width=0.6, color=colors, edgecolor="none", zorder=3)
for xi, v in zip(x, vals):
    ax.text(xi, v * 1.25, f"{v:,}", ha="center", fontsize=6.2, color="0.0")

ax.set_yscale("log")
ax.set_ylim(5, 4e4)
ax.set_xticks(x)
ax.set_xticklabels([pretty[k] for k in order], fontsize=6.2, rotation=18, ha="right")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.grid(True, axis="y", color="0.88", lw=0.5, zorder=0)
ax.set_axisbelow(True)
ax.set_ylabel("Samples (log scale)")
ax.set_title("Sample Counts by Original Label", loc="left", fontsize=7.4, pad=6)

total_sep = sum(vals[1:])
ax.text(0.98, 0.94, f"{total_sep} SEP events\nacross 4 thresholds",
        transform=ax.transAxes, ha="right", va="top", fontsize=6.0, color=TEAL)

fig.tight_layout()
fig.savefig(f"{FIG_DIR}/multiclass_distribution.pdf")
fig.savefig(f"{FIG_DIR}/multiclass_distribution.png", dpi=300)
plt.show()
print("Saved multiclass_distribution.pdf/png")
print(counts)


Saved multiclass_distribution.pdf/png
Label
NSEP     17625
gt10        85
gt100       38
gt30        27
gt60        19
Name: count, dtype: int64


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_47094/4057783772.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
